# SatQuery E0–E2 research harness
Run sequentially in a fresh Colab Python 3.11/3.12 runtime. CPU pilot first; T4 GPU for optional VQA. No full evaluation is launched here. Results are experiments, not validated application capabilities.

After a restart: rerun repository access, optional Drive, configuration and execution cells. Reinstall only if packages are missing. Identical config/source/package versions resume completed items. A changed identity creates a separate run. One writer per run folder.

In [ ]:
import sys, shutil, subprocess
print(sys.version)
print(subprocess.run(["nvidia-smi"],capture_output=True,text=True).stdout if shutil.which("nvidia-smi") else "No GPU")
print(subprocess.run(["free","-h"],capture_output=True,text=True).stdout)

## Access the code
Changes have not been pushed. Set a repository URL/ref that actually contains this research package, or upload the existing repository and set REPO to it. Never put access tokens in this notebook.

In [ ]:
from pathlib import Path
import os, subprocess
REPO_URL = ""  # Public/team repository URL containing research/, or leave empty for uploaded folder
REPO_REF = ""  # Exact commit containing the package
REPO = Path("/content/SatQuery")
if not REPO.exists():
    if not REPO_URL or not REPO_REF: raise ValueError("Upload the repository or fill REPO_URL and REPO_REF")
    subprocess.run(["git","clone",REPO_URL,str(REPO)],check=True)
    subprocess.run(["git","checkout",REPO_REF],cwd=REPO,check=True)
assert (REPO/"research/configs/pilot.yaml").exists()
os.chdir(REPO)
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"]="1"

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-r","research/requirements-colab.txt"],check=True)
print("Restart the runtime now if libraries were already imported; then skip this install cell.")

In [ ]:
USE_DRIVE = False
OUTPUT_ROOT = REPO/"research/outputs"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT=Path("/content/drive/MyDrive/SatQuery-research")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)

In [ ]:
import yaml, sys
sys.path[:0]=[str(REPO),str(REPO/"research/src")]
from satquery_research.runner import run
from satquery_research.manifests import environment
config=yaml.safe_load((REPO/"research/configs/pilot.yaml").read_text())
print(environment())
print(config)
subprocess.run([sys.executable,"-m","pytest","research/tests","-q"],check=True)

In [ ]:
run_dir=run(config,stage="prepare",output_root=OUTPUT_ROOT)
import pandas as pd, json
print(json.loads((run_dir/"dataset_audit.json").read_text()))
display(pd.read_csv(run_dir/"roster_selected.csv").head())

## Visual sanity check
Cloud opacity is continuous; reference mask is mean RGB opacity ≥ 0.20. Requested coverage is fitted approximately and achieved coverage is retained. Shadows are disabled. Thin contamination outside the binary mask is possible.

In [ ]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from satquery_research.cloud_generation import generate
roster=pd.read_csv(run_dir/"roster_selected.csv")
original=np.array(Image.open(run_dir/roster.iloc[0].original_path).convert("RGB"))
cloudy,opacity,mask,meta=generate(original,.4,config["cloud_seeds"][0],config["generator"])
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,array,title in zip(axes,[original,cloudy,opacity,mask],["Original","Synthetic cloud","Opacity","Reference mask"]):
    ax.imshow(array,cmap="gray"); ax.set_title(title); ax.axis("off")
print("Achieved coverage:",(mask>0).mean(),meta)

In [ ]:
run_dir=run(config,stage="cpu",output_root=OUTPUT_ROOT)
display(pd.read_csv(run_dir/"e0_summary.csv"))
display(Image.open(run_dir/"fig_image_gallery.png"))

## Optional small E1/E2 pilot
Enable only after reviewing the CPU outputs. Uses the same 10 development scenes, 3 questions/scene, 7 levels and 1 cloud seed: up to 1,080 predictions including clean. First use an explicitly smaller config if runtime is constrained. A T4 16 GB is the intended target, not yet verified here; CPU/insufficient memory exits gracefully. Never infer VQA accuracy from CPU repair images.

In [ ]:
ENABLE_VLM = False
if ENABLE_VLM:
    run_dir=run(config,stage="all",output_root=OUTPUT_ROOT)
else:
    print("Qwen not loaded; no VQA results generated")

In [ ]:
from satquery_research.analysis import analyze
print(analyze(run_dir))
print(json.loads((run_dir/"experiment_manifest.json").read_text()))
import shutil
archive=shutil.make_archive(str(run_dir),"zip",root_dir=run_dir)
print("Saved",archive)

## Full evaluation (not executed)
After explicit team approval and pilot/resource review:
```bash
python research/scripts/run_full.py --stage all --output-root /content/drive/MyDrive/SatQuery-research
```
This uses the held-out test partition and three fixed cloud seeds. Do not tune parameters on these results. No supported crossover should be claimed from exploratory pointwise intervals.